In [3]:
import pandas as pd
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.naive_bayes import MultinomialNB
import spacy

nltk.download('stopwords')
nltk.download('wordnet') 
nltk.download('punkt') # For using tokenization(word, sentence)
nltk.download('omw-1.4') # Requiered for lemmatization


import pandas as pd

df = pd.read_csv("/kaggle/input/spam-datasett/spam.csv", encoding = 'latin1')
df.info()

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [4]:
df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis = 1, inplace = True)

In [5]:
df.rename(columns = {'v1': 'labels', 'v2': 'message'}, inplace = True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   labels   5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


# Data Preprocessing

In [6]:
# Checking the important special characters or symbols that can contribute meaningfully to show the actual context of the message. This is because numbers or special characters like $, ! can change the meaning of the sentence.
message = df['message']

all_text = ' '.join(message)

special_characters = re.findall('[^a-zA-Z]', all_text) # Except alphabets, I am checking which special characters are there in the message
print(set(special_characters)) # Printing unique symbols using set

{'¬', ')', 'Á', '\\', '/', '©', '7', '?', 'ª', 'Â', '%', '£', '\r', '9', '"', '\x8e', '´', '\x89', '1', '<', '[', '¼', '(', '|', 'ö', 'Ð', '6', '>', '@', 'Û', 'Ô', 'È', '.', '2', ',', '+', 'Õ', '=', '_', 'å', "'", ';', 'Ì', '3', '*', '$', '0', '-', '#', ']', 'Ó', '&', '~', '8', ' ', '4', 'Ï', '5', '!', 'ä', 'Ò', '^', '÷', ':', '\x8b'}


In [7]:
# --------------------------------------
# Creating a function to lower-case
# --------------------------------------
# Creating a function to lower-case only those words which are not a proper noun, an entity, and already in uppercase. This is a required approch to preserve the meaning of the word. E.g., "US" must not be changed into "Us". Earlier, I have trained the model without using this technique by just simply lower casing all the words and the model. Now, after applying lower casing using spacy the model's recall also increased by 10%.
nlp = spacy.load('en_core_web_sm') # Loads spaCy’s small English model. 'nlp' is a pipeline that can tokenize text, tag parts of speech, and detect named entities (like countries, people, organizations).

def smart_case(sentence):
    doc = nlp(sentence) # The sentence is processed by spaCy → split into tokens (words/punctuation). Each token also has metadata like part-of-speech, is it uppercase, and named entity type. Metadata is a data about data. E.g., file content --> "Hello world" - metadata ---> file-size, file format, data-created,etc,. 2nd E.g., word - "USA" - metadata ---> is pronoun, is noun, is adjective, etc.
    processed = []

    for token in doc:
        if token.ent_type_ or token.pos_ in ["PROPN"] or token.text.isupper(): # token.ent_type_ → True if the token is part of a named entity (e.g., “US”, “NASA”, “New York”). token.pos_ in ["PROPN"] → True if it’s a proper noun (e.g., “John”, “India”). token.text.isupper() → True if the word is all uppercase (acronyms like “AI”, “NASA”).
            processed.append(token.text)
        else:
            processed.append(token.text.lower())

    return " ".join(processed)

In [8]:
lemmatizer = WordNetLemmatizer()
corpus = []

for i in range(len(df)):
    message = re.sub(r'[^a-zA-Z0-9\s!$#%@=&]', ' ', df['message'][i]) # Preserving number, alphabets along with '\s'. '\s' includes all whitespace character(tab-space, empty space, etc). \s is crucial in order to avoid the words glued together. 
    message = message.lower()
    message = message.split()
    message = [lemmatizer.lemmatize(word) for word in message if word not in set(stopwords.words('english'))]
    corpus.append(' '.join(message))


# Label Encoding

In [9]:
# Label Encoding of Target
y = np.where(df['labels'] == 'spam', 1, 0)
y.shape
print(y[:5])

[0 0 1 0 0]


# Model Training along with Tuning 'max_feature' parameter in TFidfVectorizer.

In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


for n in [3000, 3100, 3200, 3300, 3400, 3500, 4000, 5000, 6000, 7000, 8000]:
    # ------------------
    # Feature Extraction
    # ------------------
    vectorizer = TfidfVectorizer(max_features = n)
    X = vectorizer.fit_transform(corpus)


    # -------------
    # Train-Test-split
    # -------------
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42)


    # -----------
    # Model Training
    # -----------
    model = MultinomialNB() # Naive Bayes handels imbalanceness fairly since it is based on probabilities. That's why no need of balancing classes.

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"At max_feature = {n}\n    Classification Report:\n {classification_report(y_test, y_pred)}\n    and Confusion Matrix:\n   {confusion_matrix(y_test, y_pred)}\n\n") # 'mddel.score' will take X_test as input and predicts the output and compares with y_test for giving the accuracy score.
    
    

At max_feature = 3000
    Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       0.99      0.85      0.91       150

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

    and Confusion Matrix:
   [[964   1]
 [ 23 127]]


At max_feature = 3100
    Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       965
           1       0.99      0.84      0.91       150

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

    and Confusion Matrix:
   [[964   1]
 [ 24 126]]


At max_feature = 3200
    Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00

## Observation:
- ### Since at max_features = 3300, in the cofusion matrix, the False Positive = 0 which is the thing we have want the model to give as a result. Also, the         precision is at maximum value which is equal to 1 and the recall is also higher. Therefore I am using max_features = 3300.

# Training the Best Model

In [12]:
final_model = MultinomialNB

# Vecotrizer
vectorizer = TfidfVectorizer(max_features = 3300)
X = vectorizer.fit_transform(corpus).toarray()

# Train-test-split
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, random_state = 42, train_size = 0.8)

# Model-fit
model.fit(X_train2, y_train2)

# Results
y_pred2 = model.predict(X_test2)
print(f"Classification Report:\n{classification_report(y_test2, y_pred2)}\n\nConfusion Matrix: \n{confusion_matrix(y_test2, y_pred2)}")


Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       1.00      0.83      0.91       150

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115


Confusion Matrix: 
[[965   0]
 [ 25 125]]


# Saving the Best Model

In [13]:
from joblib import dump, load # Using joblib which is more efficeint than pickle for large models.

dump(model, "sms_spam_prediction_model.joblib")

dump(vectorizer, "sms_spam_prediction_vectorizer.joblib")

['sms_spam_prediction_vectorizer.joblib']